In [1]:
from pathlib import Path
import pandas as pd

In [4]:
DATA_DIR = Path('../../datasets/module_3/')
df = pd.read_csv(Path(DATA_DIR, 'feature_frame_filtered.csv'))

In [5]:
df.head()

,Unnamed: 0,variant_id,product_type,order_id,user_id,created_at,order_date,user_order_seq,outcome,ordered_before,...,count_children,count_babies,count_pets,people_ex_baby,days_since_purchase_variant_id,avg_days_to_buy_variant_id,std_days_to_buy_variant_id,days_since_purchase_product_type,avg_days_to_buy_product_type,std_days_to_buy_product_type
0,0,33826472919172,ricepastapulses,2807985930372,3482464092292,2020-10-05 16:46:19,2020-10-05 00:00:00,3,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
1,1,33826472919172,ricepastapulses,2808027644036,3466586718340,2020-10-05 17:59:51,2020-10-05 00:00:00,2,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
2,2,33826472919172,ricepastapulses,2808099078276,3481384026244,2020-10-05 20:08:53,2020-10-05 00:00:00,4,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
3,3,33826472919172,ricepastapulses,2808393957508,3291363377284,2020-10-06 08:57:59,2020-10-06 00:00:00,2,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618
4,4,33826472919172,ricepastapulses,2808429314180,3537167515780,2020-10-06 10:37:05,2020-10-06 00:00:00,3,0.0,0.0,...,0.0,0.0,0.0,2.0,33.0,42.0,31.134053,30.0,30.0,24.27618


In [33]:
# For each user, it'd be interesting to define a split based on the user_order_sequence. This will avoid data leakage for the model in production.
import pandas as pd
from typing import Tuple

def chronological_user_split(
    df: pd.DataFrame, 
    user_col: str, 
    order_col: str, 
    train_ratio: float = 0.7, 
    val_ratio: float = 0.2, 
    test_ratio: float = 0.1,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Splits a DataFrame into train, validation, and test sets chronologically per user.

    Parameters:
    - df: The input DataFrame.
    - user_col: The name of the column representing the user ID.
    - order_col: The name of the column representing the chronological sequence (e.g., user_order_seq).
    - train_ratio: Proportion of data to use for training (default: 0.7).
    - val_ratio: Proportion of data to use for validation (default: 0.15).
    - test_ratio: Proportion of data to use for testing (default: 0.15).

    Returns:
    - train_df: Training set
    - val_df: Validation set
    - test_df: Test set
    """
    assert train_ratio + val_ratio + test_ratio - 1.0 <= 0.001, "Ratios must sum to 1."

    train_list, val_list, test_list = [], [], []

    # Process each user separately
    for _, user_data in df.groupby(user_col):
        user_data = user_data.sort_values(by=order_col)

        n = len(user_data)
        train_end = int(n * train_ratio)
        val_end = train_end + int(n * val_ratio)

        train_list.append(user_data.iloc[:train_end])
        val_list.append(user_data.iloc[train_end:val_end])
        test_list.append(user_data.iloc[val_end:])

    # Combine all users' splits
    train_df = pd.concat(train_list)
    val_df = pd.concat(val_list)
    test_df = pd.concat(test_list)

    return train_df, val_df, test_df

In [34]:
train_df, val_df, test_df = chronological_user_split(df, "user_id", "user_order_seq")

0.7 0.2 0.1
